# 07 - Model Comparison

## Objective
Compare three models on the same train/test split:
1. **Logistic Regression** — explainable baseline
2. **Random Forest** — picks up non-linear relationships
3. **XGBoost** — usually strongest on tabular data

## Decision Rule
For attrition specifically, **missing a genuinely high-risk employee is expensive**.
We lean toward **recall** rather than chasing the highest accuracy.
The winner is saved as the production model.
---

In [1]:
import pandas as pd
import numpy as np
import os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_curve,
    classification_report, roc_auc_score, confusion_matrix,
    precision_score, recall_score, f1_score
)
import xgboost as xgb
import joblib
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)

DATA_PATH = "../data/processed"
MODEL_PATH = "../models"
os.makedirs(MODEL_PATH, exist_ok=True)

df = pd.read_csv(f"{DATA_PATH}/attrition_features.csv")
y = df["Attrition"]
X = df.drop(columns=["Attrition"])
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

Train: (1176, 44), Test: (294, 44)


---
## 1. Train All Three Models
---

In [2]:
# --- Logistic Regression (with scaling) ---
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_sc, y_train)
y_prob_lr = lr.predict_proba(X_test_sc)[:, 1]
y_pred_lr = lr.predict(X_test_sc)
print("Logistic Regression trained.")

# --- Random Forest ---
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_prob_rf = rf.predict_proba(X_test)[:, 1]
y_pred_rf = rf.predict(X_test)
print("Random Forest trained.")

# --- XGBoost ---
xgb_model = xgb.XGBClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
    random_state=42, eval_metric="logloss", use_label_encoder=False
)
xgb_model.fit(X_train, y_train)
y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]
y_pred_xgb = xgb_model.predict(X_test)
print("XGBoost trained.")

Logistic Regression trained.


Random Forest trained.


XGBoost trained.


---
## 2. Build Comparison Table
---

In [3]:
results = []
for name, y_pred, y_prob in [
    ("Logistic Regression", y_pred_lr, y_prob_lr),
    ("Random Forest", y_pred_rf, y_prob_rf),
    ("XGBoost", y_pred_xgb, y_prob_xgb),
]:
    results.append({
        "Model": name,
        "Precision": round(precision_score(y_test, y_pred), 4),
        "Recall": round(recall_score(y_test, y_pred), 4),
        "F1": round(f1_score(y_test, y_pred), 4),
        "ROC-AUC": round(roc_auc_score(y_test, y_prob), 4),
    })

results_df = pd.DataFrame(results)
print("=== MODEL COMPARISON ===")
print(results_df.to_string(index=False))

=== MODEL COMPARISON ===
              Model  Precision  Recall     F1  ROC-AUC
Logistic Regression     0.6296  0.3617 0.4595   0.8192
      Random Forest     0.4615  0.1277 0.2000   0.8259
            XGBoost     0.5758  0.4043 0.4750   0.7980


---
## 3. Visualize Comparison
---

In [4]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart of metrics
metrics = ["Precision", "Recall", "F1", "ROC-AUC"]
x = np.arange(len(metrics))
width = 0.25
colors = ["#3b82f6", "#22c55e", "#ef4444"]

for i, (_, row) in enumerate(results_df.iterrows()):
    vals = [row[m] for m in metrics]
    axes[0].bar(x + i * width, vals, width, label=row["Model"], color=colors[i])

axes[0].set_xticks(x + width)
axes[0].set_xticklabels(metrics)
axes[0].set_ylim(0, 1)
axes[0].set_title("Model Comparison — Key Metrics")
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis="y")

# ROC curves
for name, y_prob, color in [
    ("Logistic Regression", y_prob_lr, colors[0]),
    ("Random Forest", y_prob_rf, colors[1]),
    ("XGBoost", y_prob_xgb, colors[2]),
]:
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    axes[1].plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})", color=color)

axes[1].plot([0, 1], [0, 1], "k--", alpha=0.5)
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("ROC Curves")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{MODEL_PATH}/model_comparison.png", dpi=100, bbox_inches="tight")
plt.show()
print("Saved: models/model_comparison.png")

Saved: models/model_comparison.png


---
## 4. Select Winner and Save
---

In [5]:
# Pick winner based on F1 (balance of precision and recall for imbalanced data)
winner_idx = results_df["F1"].idxmax()
winner_name = results_df.loc[winner_idx, "Model"]
winner_f1 = results_df.loc[winner_idx, "F1"]
winner_auc = results_df.loc[winner_idx, "ROC-AUC"]
winner_recall = results_df.loc[winner_idx, "Recall"]

print(f"Winner: {winner_name}")
print(f"  F1:      {winner_f1}")
print(f"  Recall:  {winner_recall}")
print(f"  ROC-AUC: {winner_auc}")

# Map model name to object
model_map = {
    "Logistic Regression": (lr, scaler),
    "Random Forest": (rf, None),
    "XGBoost": (xgb_model, None),
}
winner_model, winner_scaler = model_map[winner_name]

# Save pipeline
pipeline = {
    "model": winner_model,
    "scaler": winner_scaler,
    "feature_names": list(X.columns),
    "model_name": winner_name,
}
joblib.dump(pipeline, f"{MODEL_PATH}/attrition_pipeline.joblib")
print(f"\nSaved: models/attrition_pipeline.joblib")

Winner: XGBoost
  F1:      0.475
  Recall:  0.4043
  ROC-AUC: 0.798

Saved: models/attrition_pipeline.joblib


---
## Model Comparison Summary

| Model | Precision | Recall | F1 | ROC-AUC |
| --- | ---: | ---: | ---: | ---: |

In [6]:
# Print final comparison
print(results_df.to_markdown(index=False))

| Model               |   Precision |   Recall |     F1 |   ROC-AUC |
|:--------------------|------------:|---------:|-------:|----------:|
| Logistic Regression |      0.6296 |   0.3617 | 0.4595 |    0.8192 |
| Random Forest       |      0.4615 |   0.1277 | 0.2    |    0.8259 |
| XGBoost             |      0.5758 |   0.4043 | 0.475  |    0.798  |



**Winner selected** and saved as `models/attrition_pipeline.joblib`.

**Next step:** Model Explainability (08_model_explainability.ipynb)
